In [63]:
import pandas as pd

train = pd.read_csv('train.csv')
print(train.shape)
print(train['avg_delay_minutes_next_30m'].describe())


(250000, 94)
count    250000.000000
mean         18.962296
std          27.351374
min           0.000000
25%           4.278801
50%           9.032652
75%          25.791869
max         715.858119
Name: avg_delay_minutes_next_30m, dtype: float64


In [64]:
print(train.isnull().sum())
print(train.dtypes.value_counts())

ID                              0
layout_id                       0
scenario_id                     0
order_inflow_15m            29564
unique_sku_15m              29924
                            ...  
dock_to_stock_hours         29468
kpi_otd_pct                 29405
backorder_ratio             29552
shift_handover_delay_min    29534
sort_accuracy_pct           29770
Length: 94, dtype: int64
float64    88
object      3
int64       3
Name: count, dtype: int64


In [65]:
train_scenarios = set(train['scenario_id'].unique())

test = pd.read_csv('test.csv')
test_scenarios = set(test['scenario_id'].unique())

overlap = train_scenarios & test_scenarios
print("train 시나리오 수: ", len(train_scenarios))
print("test 시나리오 수: ", len(test_scenarios))
print("겹치는 시나리오 수: ", len(overlap))

train 시나리오 수:  10000
test 시나리오 수:  2000
겹치는 시나리오 수:  0


In [66]:
train_layouts = set(train['layout_id'].unique())
test_layouts = set(test['layout_id'].unique())

overlap = train_layouts & test_layouts

print("train layout 수:", len(train_layouts))
print("test layout 수:", len(test_layouts))
print("겹치는 layout 수:", len(overlap))

train layout 수: 250
test layout 수: 100
겹치는 layout 수: 50


In [67]:
train = train.drop(['ID','layout_id','scenario_id'], axis=1)

In [68]:
print(train.shape)

(250000, 91)


In [69]:
X = train.drop('avg_delay_minutes_next_30m', axis=1)
y = train['avg_delay_minutes_next_30m']


In [70]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape)

(200000, 90) (50000, 90)


In [71]:
from lightgbm import LGBMRegressor
import optuna
import numpy as np
from sklearn.metrics import mean_absolute_error

def objective(trial):
    params = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'verbose': -1
    }
    
    model = LGBMRegressor(**params)
    model.fit(X_train, np.log1p(y_train))  # 로그 변환 유지
    
    pred = np.expm1(model.predict(X_val))
    mae = mean_absolute_error(y_val, pred)
    return mae

study = optuna.create_study(direction='minimize')  # MAE는 낮을수록 좋음
study.optimize(objective, n_trials=30)

print('최적 파라미터:', study.best_params)
print('최적 MAE:', study.best_value)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-29 12:47:41,883] A new study created in memory with name: no-name-41674294-e182-453d-a6a5-833d5b6ed27a
[I 2026-06-29 12:47:57,132] Trial 0 finished with value: 8.828145424894206 and parameters: {'learning_rate': 0.03202038515498759, 'num_leaves': 112, 'max_depth': 6, 'min_child_samples': 55, 'subsample': 0.6264811114672467, 'colsample_bytree': 0.6062702197268213}. Best is trial 0 with value: 8.828145424894206.
[I 2026-06-29 12:48:04,414] Trial 1 finished with value: 9.320851540024842 and parameters: {'learning_rate': 0.014325787481520203, 'num_leaves': 66, 'max_depth': 4, 'min_child_samples': 92, 'subsample': 0.7796869210812353, 'colsample_bytree': 0.9243971127419464}. Best is trial 0 with value

최적 파라미터: {'learning_rate': 0.09722819993822204, 'num_leaves': 135, 'max_depth': 10, 'min_child_samples': 45, 'subsample': 0.9695683556233664, 'colsample_bytree': 0.917526518006572}
최적 MAE: 7.844444823346267


In [73]:
from sklearn.model_selection import KFold
import numpy as np

best_params = study.best_params 
best_params['n_estimators'] = 1000
best_params['random_state'] = 42
best_params['verbose'] = -1


kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []

for train_idx, val_idx in kf.split(X):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    m = LGBMRegressor(**best_params)
    m.fit(X_tr, np.log1p(y_tr))
    pred = np.expm1(m.predict(X_va))  # 로그 되돌리기
    
    mae = mean_absolute_error(y_va, pred)
    fold_maes.append(mae)
    print(f'Fold MAE: {mae:.2f}')

print(f'\n평균 MAE: {np.mean(fold_maes):.2f}, 표준편차: {np.std(fold_maes):.2f}')

Fold MAE: 7.84
Fold MAE: 7.88
Fold MAE: 7.83
Fold MAE: 7.90
Fold MAE: 7.90

평균 MAE: 7.87, 표준편차: 0.03


In [74]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor

best_params = study.best_params
best_params['n_estimators'] = 1000
best_params['random_state'] = 42
best_params['verbose'] = -1

final_model = LGBMRegressor(**best_params)
final_model.fit(X, np.log1p(y))

test = pd.read_csv('test.csv')
test_ids = test['ID']
X_test = test.drop(['ID', 'layout_id', 'scenario_id'], axis=1)

pred = np.expm1(final_model.predict(X_test))
pred = np.clip(pred, 0, None)

submission = pd.DataFrame({
    'ID': test_ids,
    'avg_delay_minutes_next_30m': pred
})
submission.to_csv('my_submission_tuned.csv', index=False)
print(submission.head())
print(submission.shape)

            ID  avg_delay_minutes_next_30m
0  TEST_000000                   13.707726
1  TEST_000001                   14.216570
2  TEST_000002                   14.421265
3  TEST_000003                   12.110241
4  TEST_000004                   16.857683
(50000, 2)
